In [5]:
import pandas as pd
import numpy as np
import geopandas as gpd
import re

INPUT_CSV = 'csv_input/07290012_modified_shapefile.csv'
OUTPUT_CSV = 'csv_input/07290012_renamed.csv'

df = pd.read_csv(INPUT_CSV)
# df = df[['County', 'Shape_ID', 'geometry']]

gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geometry']))
gdf['centroid'] = gdf.geometry.centroid

county_order = df['County'].unique().tolist()

In [6]:
# Compute each county's centroid as the area-weighted average of its polygon centroids
# (avoids unary_union, which requires topologically valid geometries)
def weighted_county_centroid(group):
    areas = group.geometry.area
    total_area = areas.sum()
    cx = (group['centroid'].x * areas).sum() / total_area
    cy = (group['centroid'].y * areas).sum() / total_area
    return cx, cy

county_centroids = (
    gdf.groupby('County', observed=True)
    .apply(weighted_county_centroid, include_groups=False)
)

def assign_clockwise_ids(group, county_centroid):
    cx, cy = county_centroid

    group = group.copy()
    group['_dx'] = group['centroid'].x - cx
    group['_dy'] = group['centroid'].y - cy
    group['Dist_to_Centroid'] = np.hypot(group['_dx'], group['_dy'])
    group['_angle'] = np.arctan2(group['_dy'], group['_dx'])  # counter-clockwise radians

    # _001 = nearest to county centroid
    nearest_idx = group['Dist_to_Centroid'].idxmin()
    start_angle = group.loc[nearest_idx, '_angle']

    # Clockwise: negate angles; shift so start_angle = 0; wrap to [0, 2pi)
    group['_cw_angle'] = (-(group['_angle'] - start_angle)) % (2 * np.pi)

    # Nearest polygon gets 0 -> sorts first
    group = group.sort_values('_cw_angle').reset_index(drop=True)

    # Assign Shape_ID
    county = group['County'].iloc[0]
    group['Shape_ID'] = [f"{county}_{i+1:03d}" for i in range(len(group))]

    return group.drop(columns=['_dx', '_dy', '_angle', '_cw_angle'])

gdf['County'] = pd.Categorical(gdf['County'], categories=county_order, ordered=True)

result_parts = []
for county in county_order:
    group = gdf[gdf['County'] == county]
    if group.empty:
        continue
    numbered = assign_clockwise_ids(group, county_centroids[county])
    result_parts.append(numbered)

gdf = pd.concat(result_parts, ignore_index=True)

gdf

,Zone,County,District,Shape_ID,geometry,Area,Population_Density,Region,Location,County_Type,Capacity,Population,District_Ward,Clusters,Constituency,centroid,Dist_to_Centroid
0,Alington,A01,NaN,A01_001,"POLYGON ((2883.27 1328.07, 2878.63 1333.14, 28...",0.577635,16006,Alma Valley East,Inner City,E1,1.04,10428,NaN,Creative_Rich,NaN,POINT (2879.083 1327.761),4.051879
1,Alington,A01,NaN,A01_002,"POLYGON ((2866.75 1312, 2864.7 1312.61, 2866.4...",1.008987,10812,Alma Valley East,Inner City,A1,1.00,10036,NaN,Posh,NaN,POINT (2876.535 1313.657),18.368155
2,Alington,A01,NaN,A01_003,"POLYGON ((2876.55 1317.07, 2879.48 1318.86, 28...",0.416791,13038,Alma Valley East,Inner City,F1,1.05,6029,NaN,Creative_Rich,NaN,POINT (2877.455 1320.112),11.848894
3,Alington,A01,NaN,A01_004,"POLYGON ((2869.47 1318.22, 2869.13 1316.06, 28...",0.386001,17278,Alma Valley East,Inner City,D1,1.00,7069,NaN,Creative_Rich,NaN,POINT (2872.653 1317.382),15.918833
4,Alington,A01,NaN,A01_005,"POLYGON ((2878.04 1323.18, 2875.08 1326.19, 28...",0.327799,19928,Alma Valley East,Inner City,E1,0.89,5768,NaN,Creative_Rich,NaN,POINT (2874.375 1323.444),9.750398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20845,Waterstone,WA06,Aqua Corner,WA06_001,"POLYGON ((1365 660, 1365 569, 1365 478, 1456 4...",258.135508,11,South East,Rural,F1,0.91,2617,NaN,NaN,NaN,POINT (1410.5 569),0.000000
20846,Waterstone,WA07,Aqua Corner,WA07_001,"POLYGON ((1273 569, 1273 478, 1365 478, 1365 5...",130.486081,21,South East,Rural,F1,1.00,2615,NaN,NaN,NaN,POINT (1319 523.5),0.000000
20847,Waterstone,WA08,Aqua Corner,WA08_001,"POLYGON ((1273 569, 1273 660, 1365 660, 1365 5...",130.486081,21,South East,Rural,F1,0.86,2584,NaN,NaN,NaN,POINT (1319 614.5),0.000000
20848,Waterstone,WA09,Aqua Corner,WA09_001,"POLYGON ((1456 660, 1365 660, 1273 660, 1273 8...",766.800550,3,South East,Rural,F1,0.92,2084,NaN,NaN,NaN,POINT (1388.635 785.057),0.000000


In [7]:
print("=== Shape_ID Validation ===\n")
all_valid = True

for county in county_order:
    group = gdf[gdf['County'] == county]
    ids = group['Shape_ID'].tolist()
    n = len(ids)
    expected = [f"{county}_{i:03d}" for i in range(1, n + 1)]

    bad_format = [s for s in ids if not re.fullmatch(rf"{re.escape(county)}_\d{{3}}", s)]
    missing = set(expected) - set(ids)
    duplicates = [s for s in ids if ids.count(s) > 1]

    if bad_format or missing or duplicates:
        all_valid = False
        print(f"[FAIL] {county} ({n} polygons)")
        if bad_format:   print(f"  Bad format:  {bad_format}")
        if missing:      print(f"  Missing IDs: {sorted(missing)}")
        if duplicates:   print(f"  Duplicates:  {list(set(duplicates))}")
    else:
        print(f"[ OK ] {county} \u2014 {county}_001 to {county}_{n:03d}")

print("\nAll Shape_IDs valid." if all_valid else "\nIssues found \u2014 review above.")

=== Shape_ID Validation ===

[ OK ] A01 — A01_001 to A01_071
[ OK ] A02 — A02_001 to A02_012
[ OK ] A03 — A03_001 to A03_041
[ OK ] A04 — A04_001 to A04_054
[ OK ] A05 — A05_001 to A05_060
[ OK ] A06 — A06_001 to A06_063
[ OK ] A07 — A07_001 to A07_091
[ OK ] A08 — A08_001 to A08_082
[ OK ] A09 — A09_001 to A09_085
[ OK ] A10 — A10_001 to A10_217
[ OK ] A11 — A11_001 to A11_201
[ OK ] A12 — A12_001 to A12_178
[ OK ] C01 — C01_001 to C01_043
[ OK ] C02 — C02_001 to C02_046
[ OK ] C03 — C03_001 to C03_129
[ OK ] C04 — C04_001 to C04_056
[ OK ] C05 — C05_001 to C05_060
[ OK ] C06 — C06_001 to C06_075
[ OK ] C07 — C07_001 to C07_060
[ OK ] C08 — C08_001 to C08_049
[ OK ] C09 — C09_001 to C09_043
[ OK ] C10 — C10_001 to C10_128
[ OK ] C11 — C11_001 to C11_076
[ OK ] C12 — C12_001 to C12_085
[ OK ] C13 — C13_001 to C13_032
[ OK ] C14 — C14_001 to C14_090
[ OK ] C15 — C15_001 to C15_089
[ OK ] C16 — C16_001 to C16_067
[ OK ] C17 — C17_001 to C17_065
[ OK ] C18 — C18_001 to C18_062
[ OK ] C19 

In [8]:
gdf.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(gdf)} rows to {OUTPUT_CSV}")

Saved 20850 rows to csv_input/07290012_renamed.csv
